In [0]:
%sql


select distinct CMS.Report_ID
from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as WAA
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cms_file_metadata as CMS
on WAA.WEBI_Doc_ID = CMS.Report_ID
where CMS.Report_ID is not null
and CMS.scheduled=true
order by CMS.Report_ID desc;

select count(*) from _sqldf

In [0]:
%sql

select count(distinct Document_Id) from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata;

select distinct FID.Report_ID  
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.bo_user_access_audit as WEBA
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.sapbo_full_doc_id as FID
on upper(FID.Report_CUID) =upper(WEBA.Object_ID)
where WEBA.Object_ID is not null;

CREATE OR REPLACE TABLE dataplatform01_central_dev_catalog_01.custom_sap_bo.Webi_schedule_scan_tracker
select distinct FID.Report_ID, 
(Case when WEBA.Object_ID is not null then "Active" end) as Audit, 
CMS.scheduled as CMS_scheduled,
"" as Scanned,
"" as Scanned_Date,
"" as Result
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.sapbo_full_doc_id as FID
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.bo_user_access_audit as WEBA
on upper(FID.Report_CUID) =upper(WEBA.Object_ID)
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as CMS
on upper(FID.Report_ID) = upper(CMS.Document_Id)
where upper(FID.Report_Type)="WEBI"
order by FID.Report_ID asc;


In [0]:
%sql
select Audit, CMS_scheduled, count(distinct Report_ID) as cnt from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_scan_tracker
group by Audit, CMS_scheduled


Below to use create the table for Schedule reports scan output, only the ones has schedule and with a record in the instance got captured. 

In [0]:
%sql

CREATE TABLE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details (
    -- Document identifiers
    document_id                VARCHAR(64) NOT NULL,
    document_cuid              VARCHAR(64),
    document_name              VARCHAR(512),
    folder_id                  VARCHAR(64),
    full_path                  VARCHAR(1024),

    -- Document metadata
    document_updated_ts        TIMESTAMP,
    document_scheduled         VARCHAR(64),
    document_created_by        VARCHAR(256),
    document_last_author       VARCHAR(256),

    -- Schedule identifiers
    schedule_id                VARCHAR(64),
    schedule_name              VARCHAR(512),

    -- Schedule instance metadata
    schedule_updated_ts        TIMESTAMP,
    schedule_format            VARCHAR(64),
    schedule_status            VARCHAR(64),

    -- ✅ Key motivation fields
    repetition_type            VARCHAR(32),  -- once | hourly | daily | weekly | monthly | calendar
    repetition                 STRING,          -- structured repetition block

    destination_type           VARCHAR(32),  -- mail | inbox | file | ftp
    destination                STRING,          -- structured destination block

    parameters                 STRING,          -- schedule parameters
    Instance_json              STRING,          -- schedule full json API respons
    -- Extraction / execution stats  
    ingestion_ts               TIMESTAMP
    
    );




In [0]:
%sql

select SST.Audit, SST.CMS_scheduled, count(distinct SST.Report_ID) from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_scan_tracker as SST
-- where SST.Report_ID="143617"
group by SST.Audit,SST.CMS_scheduled
order by SST.CMS_scheduled asc


In [0]:
%sql
select distinct destination_type, destination from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details
where destination_type is null;

select distinct repetition_type, repetition from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details
where repetition_type is null;

In [0]:
%sql
select 
  destination_type, 
  count(distinct document_id) as cnt,
  concat(format_number(100.0 * count(distinct document_id) / sum(count(distinct document_id)) over (), 1), '%') as pct_of_total
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details
where  schedule_status="Completed"
group by  destination_type
order by cnt desc
;

select destination_type, destination from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details
where schedule_status="Completed"
and  destination_type ="filesystem"
order by

In [0]:
%sql
select 
  date_format(date_trunc('month', schedule_updated_ts), 'yyyy-MM-dd') as schedule_month,
  count(distinct document_id) as cnt,
  concat(format_number(100.0 * count(distinct document_id) / sum(count(distinct document_id)) over (), 1), '%') as pct_of_total
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details
where schedule_status = 'Completed'
group by schedule_month
order by schedule_month desc

In [0]:
%sql
select distinct file_path from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched

In [0]:
%sql
select * from  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details